# SN-04 — Sirenisation Phase 3 (matching approfondi APE + date)

Sur les EJ encore non résolus en Phase 2 (DOUTEUX, REJETE, SANS_CANDIDAT au rang 1), on applique un scoring **approfondi** intégrant :
- code APE (`cdape_stru` ↔ `activitePrincipaleUniteLegale`)
- année de création (`dtouverture_stru` ↔ `dateCreationUniteLegale`)

**Score combiné** :
- APE et date dispos : `0.70 × global + 0.20 × ape + 0.10 × date`
- APE seul           : `0.80 × global + 0.20 × ape`
- date seule         : `0.80 × global + 0.20 × date`
- aucun              : `score_global` standard

Blocking élargi : par **département** (au lieu de la commune) pour augmenter le rappel.

In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('../..'))

import pandas as pd
from tqdm.auto import tqdm

from src.sirenisation import scorer_paire_approfondi
from src.matching     import classifier_resultat
from src.excel_export import export_topn_excel, LABELS
from src.display      import afficher_tableau, afficher_synthese
from config.settings  import (
    SN_PERIMETRE, SIRENE_UL_CLEAN, SN_PHASE1, SN_PHASE2, SN_PHASE3, RESULTS_SN_DIR,
)
RESULTS_SN_DIR.mkdir(parents=True, exist_ok=True)

# Récupération des SIREN validés en P1 (pour désactivation bonus sur EJ jumeaux)
FEUILLES_VALIDES = {'Valide_fort', 'Valide'}
sheets_p1 = pd.read_excel(SN_PHASE1, sheet_name=None, dtype=str)
sirens_valides_p1 = set()
for nom, sdf in sheets_p1.items():
    if nom in FEUILLES_VALIDES and 'nmsiren_stru' in sdf.columns:
        sirens_valides_p1.update(
            sdf['nmsiren_stru'].dropna().astype(str)
            .str.replace(r'\s', '', regex=True).str.strip()
        )
sirens_valides_p1.discard('')
print(f'SIREN validés en P1 (pour désactivation bonus) : {len(sirens_valides_p1):,}')

SIREN validés en P1 (pour désactivation bonus) : 38,154


## 1. Récupération des EJ non résolus en Phase 2

In [2]:
NON_RES = {'DOUTEUX', 'REJETE', 'SANS_CANDIDAT'}

df_p2 = pd.read_excel(SN_PHASE2, sheet_name='Top5', dtype=str)
df_p2['rang'] = pd.to_numeric(df_p2['rang'], errors='coerce')
r1 = df_p2[df_p2['rang'] == 1]
ids_phase3 = set(r1[r1['statut_candidat'].isin(NON_RES)]['idstructure_stru']
                  .dropna().astype(str).tolist())

df_perimetre = pd.read_parquet(SN_PERIMETRE)
df_perimetre['idstructure_stru'] = df_perimetre['idstructure_stru'].astype(str)
df_phase3 = df_perimetre[df_perimetre['idstructure_stru'].isin(ids_phase3)].copy().reset_index(drop=True)

df_ul = pd.read_parquet(SIRENE_UL_CLEAN)
df_ul['siren'] = df_ul['siren'].astype(str)

# Pré-calculs sur df_ul (évite les conversions à chaque paire)
from src.sirenisation import normaliser_ape
df_ul['_ape_norm']      = df_ul['activitePrincipaleUniteLegale'].apply(normaliser_ape)
df_ul['_ape_div']       = df_ul['_ape_norm'].str[:2]   # division APE (2 premiers car.)
df_ul['_annee_ul']      = pd.to_numeric(
    df_ul['dateCreationUniteLegale'].astype(str).str[:4], errors='coerce'
)

# Idem côté EJ pour le filtrage rapide
df_phase3['_ape_norm'] = df_phase3['cdape_stru'].apply(normaliser_ape)
df_phase3['_ape_div']  = df_phase3['_ape_norm'].str[:2]
df_phase3['_annee_ej'] = pd.to_numeric(
    df_phase3['dtouvertstruct_stru'].astype(str).str[:4], errors='coerce'
)

print(f'EJ à traiter Phase 3 : {len(df_phase3):,}')
print(f'  cdape_stru présent      : {df_phase3["_ape_norm"].ne("").sum():,}')
print(f'  dtouvertstruct_stru présent: {df_phase3["_annee_ej"].notna().sum():,}')
print(f'UL SIRENE candidates  : {len(df_ul):,}')

EJ à traiter Phase 3 : 3,118
  cdape_stru présent      : 1,190
  dtouvertstruct_stru présent: 3,118
UL SIRENE candidates  : 15,184,224


## 2. Indexation des UL par département

In [3]:
ul_par_dept = df_ul.groupby('dept_ul', sort=False)
print(f'Départements avec au moins une UL : {ul_par_dept.ngroups:,}')

Départements avec au moins une UL : 108


## 3. Matching approfondi top 3

In [ ]:
BONUS_SIREN_COHERENT = 15.0

lignes_top3 = []
lignes_orphelins = []

for _, row_ej in tqdm(df_phase3.iterrows(), total=len(df_phase3),
                       desc='Phase 3 matching approfondi'):
    dept = row_ej.get('dept_ej')
    siren_ej = str(row_ej.get('nmsiren_stru', '') or '').strip()
    bonus_applicable = (siren_ej != '') and (siren_ej not in sirens_valides_p1)

    if not dept or dept not in ul_par_dept.groups:
        lignes_orphelins.append({
            **row_ej.to_dict(),
            'siren_ref_app': None, 'nom_ul_retenu': None,
            'score_nom': None, 'score_adresse': None, 'score_global': None,
            'score_ape': None, 'score_date': None,
            'ape_ej': None, 'ape_ul': None,
            'annee_creation_ej': None, 'annee_creation_ul': None,
            'score_global_approfondi': None,
            'siren_coherent': False, 'bonus_applique': False,
            'score_approfondi_ajuste': None,
            'rang': 1, 'statut_candidat': 'SANS_CANDIDAT',
        })
        continue

    candidates = ul_par_dept.get_group(dept)

    # PRÉ-FILTRE APE : ne garder que les UL avec APE compatible (même division 2 car.)
    # ou les UL qui ont un SIREN cohérent (toujours évaluées même hors filtre APE)
    ape_div_ej = row_ej.get('_ape_div', '')
    if ape_div_ej:
        mask_filtre = (
            (candidates['_ape_div'] == ape_div_ej)   
            | (candidates['_ape_div'] == '')         
            | (candidates['siren'] == siren_ej)      
        )
        candidates = candidates[mask_filtre]

    if len(candidates) == 0:
        lignes_orphelins.append({
            **row_ej.to_dict(),
            'siren_ref_app': None, 'nom_ul_retenu': None,
            'score_nom': None, 'score_adresse': None, 'score_global': None,
            'score_ape': None, 'score_date': None,
            'ape_ej': None, 'ape_ul': None,
            'annee_creation_ej': None, 'annee_creation_ul': None,
            'score_global_approfondi': None,
            'siren_coherent': False, 'bonus_applique': False,
            'score_approfondi_ajuste': None,
            'rang': 1, 'statut_candidat': 'SANS_CANDIDAT',
        })
        continue

    scores = []
    for _, row_ul in candidates.iterrows():
        s = scorer_paire_approfondi(row_ej, row_ul)
        siren_ul = str(row_ul['siren']).strip()
        coherent = bool(siren_ej) and (siren_ej == siren_ul)
        bonus = BONUS_SIREN_COHERENT if (coherent and bonus_applicable) else 0.0
        score_ajuste = min(s['score_global_approfondi'] + bonus, 100.0)

        scores.append({
            'siren_ref_app':                 row_ul['siren'],
            'denominationUniteLegale':       row_ul.get('denominationUniteLegale'),
            'sigleUniteLegale':              row_ul.get('sigleUniteLegale'),
            'adresse_siege_complete_ul':     row_ul.get('adresse_siege_complete_ul'),
            'codeCommuneEtablissement':      row_ul.get('codeCommuneEtablissement'),
            'categorieJuridiqueUniteLegale': row_ul.get('categorieJuridiqueUniteLegale'),
            'activitePrincipaleUniteLegale': row_ul.get('activitePrincipaleUniteLegale'),
            'dateCreationUniteLegale':       row_ul.get('dateCreationUniteLegale'),
            **s,
            'siren_coherent':           coherent,
            'bonus_applique':           bool(bonus > 0),
            'score_approfondi_ajuste':  round(score_ajuste, 2),
        })

    scores.sort(key=lambda x: x['score_approfondi_ajuste'], reverse=True)
    for rang, sc in enumerate(scores[:3], start=1):
        statut = classifier_resultat(
            sc['score_approfondi_ajuste'], sc['score_nom'], sc['score_adresse']
        )
        lignes_top3.append({
            **row_ej.to_dict(), **sc,
            'rang': rang, 'statut_candidat': statut,
        })

df_top3 = pd.DataFrame(lignes_top3 + lignes_orphelins)

# Nettoyage : on retire les colonnes techniques préfixées par _
df_top3 = df_top3.loc[:, ~df_top3.columns.str.startswith('_')]

n_coherents_top1   = df_top3[(df_top3['rang'] == 1) & (df_top3['siren_coherent'])].shape[0]
n_bonus_applique   = df_top3[df_top3.get('bonus_applique', False) == True].shape[0]
n_jumeaux_protegés = (
    df_top3[(df_top3['siren_coherent']) & (df_top3.get('bonus_applique', False) == False)].shape[0]
)
print(f'\nLignes top 3 Phase 3 : {len(df_top3):,}')
print(f'Candidats avec SIREN cohérent au rang 1     : {n_coherents_top1:,}')
print(f'Lignes avec bonus +15 appliqué               : {n_bonus_applique:,}')
print(f'Lignes "EJ jumeaux" (cohérent mais pas bonus): {n_jumeaux_protegés:,}')
print()
print(df_top3[df_top3['rang'] == 1]['statut_candidat'].value_counts())

Phase 3 matching approfondi:   0%|          | 0/3118 [00:00<?, ?it/s]


Lignes top 3 Phase 3 : 9,354
Candidats avec SIREN cohérent au rang 1     : 1,439
Lignes avec bonus +15 appliqué               : 1,640
Lignes "EJ jumeaux" (cohérent mais pas bonus): 3

statut_candidat
DOUTEUX        2098
VALIDE          971
REJETE           46
VALIDE_FORT       3
Name: count, dtype: int64


## 4. Aperçu

In [5]:
afficher_tableau(
    df_top3[['idstructure_stru', 'raisonsociale_stru',
             'siren_ref_app', 'denominationUniteLegale', 'nom_ul_retenu',
             'ape_ej', 'ape_ul', 'score_ape',
             'annee_creation_ej', 'annee_creation_ul', 'score_date',
             'score_nom', 'score_adresse', 'score_global',
             'score_global_approfondi', 'statut_candidat', 'rang']],
    'Aperçu top 3 Phase 3', max_lignes=9,
)

idstructure_stru,raisonsociale_stru,siren_ref_app,denominationUniteLegale,nom_ul_retenu,ape_ej,ape_ul,score_ape,annee_creation_ej,annee_creation_ul,score_date,score_nom,score_adresse,score_global,score_global_approfondi,statut_candidat,rang
1843328,CCAS BARJAC,263003873,CENTRE COMMUNAL D'ACTION SOCIALE,CCAS,8899B,8899B,100.000000,2001,1986,20.000000,57.870000,30.000000,41.150000,50.800000,DOUTEUX,1
1843328,CCAS BARJAC,441394483,LA CELLE,CELLE,8899B,8899B,100.000000,2001,2001,100.000000,31.170000,40.000000,36.470000,55.530000,DOUTEUX,2
1843328,CCAS BARJAC,200103232,CENTRE COMMUNAL D'ACTION SOCIALE DE SAINT-ANDRE-DE-ROQUEPERTUIS,CCAS,8899B,8899B,100.000000,2001,2024,0.000000,57.870000,40.000000,47.150000,53.000000,DOUTEUX,3
1843342,PHARMACIE LABEILLE,838457927,KINEVA,KINEVA,,6619A,nan,2018,2018,100.000000,9.000000,70.000000,45.600000,56.480000,DOUTEUX,1
1843342,PHARMACIE LABEILLE,879474088,LILY-ROSE,LILY ROSE,,6820B,nan,2018,2019,100.000000,30.670000,52.480000,43.750000,55.000000,DOUTEUX,2
1843342,PHARMACIE LABEILLE,880443494,LES ARCEAUX DE PROVENCE,ARCEAUX PROVENCE,,6820B,nan,2018,2019,100.000000,34.820000,47.890000,42.670000,54.140000,DOUTEUX,3
1843347,ASSOC CHATEAU SILHOL,527677769,ASSOCIATION PAROISSIALE CATHOLIQUE DE SUMENE,PAROISSIALE CATHOLIQUE SUMENE,,9499Z,nan,2005,2004,100.000000,41.360000,40.000000,40.540000,52.430000,DOUTEUX,1
1843347,ASSOC CHATEAU SILHOL,492784897,AMICALE SPORTIVE DE LEDIGNAN,AMICALE SPORTIVE LEDIGNAN,,8551Z,nan,2005,2006,100.000000,39.470000,40.000000,39.790000,51.830000,DOUTEUX,2
1843347,ASSOC CHATEAU SILHOL,484867817,ASS ATELIERS EDUCATIFS ET SOLIDARITE,ASS ATELIERS EDUCATIFS SOLIDARITE,,8899B,nan,2005,2005,100.000000,39.160000,40.000000,39.660000,51.730000,DOUTEUX,3


## 5. Export Excel

In [6]:
COLS_EXPORT = [
    'idstructure_stru', 'nmfinessej_stru', 'nmfinessetab_stru', 
    'categetab_stru', 'nmsiren_stru', 'raisonsociale_stru',
    'cdcommune_stru', 'adresse_complete_ej', 'sous_ensemble',
    'cdape_stru', 'dtouvertstruct_stru',
    'siren_ref_app', 'siren_coherent', 'bonus_applique',
    'denominationUniteLegale', 'sigleUniteLegale',
    'nom_ul_retenu', 'adresse_siege_complete_ul', 'codeCommuneEtablissement',
    'categorieJuridiqueUniteLegale',
    'ape_ej', 'ape_ul', 'score_ape',
    'annee_creation_ej', 'annee_creation_ul', 'score_date',
    'score_nom', 'score_adresse', 'score_global',
    'score_global_approfondi', 'score_approfondi_ajuste',
    'statut_candidat', 'rang',
]

df_top3['rang'] = df_top3['rang'].astype(int)
compteurs = export_topn_excel(df_top3, SN_PHASE3, COLS_EXPORT, sheet_name='Top3')

afficher_synthese({LABELS[s]: n for s, n in compteurs.items()},
                  'Synthèse Phase 3 — matching approfondi (rang 1)')
print(f'\nFichier : {SN_PHASE3}')

Statut,Nb,% du total
Valide_fort,3,0.1%
Valide,971,31.1%
Douteux,"2,098",67.3%
Rejeté,46,1.5%
Sans_candidat,0,0.0%
TOTAL,"3,118",100.0%



Fichier : /home/jovyan/work/projet_finess_sirene/results/sirenisation/sirenisation_phase3_approfondi.xlsx
